# Granite Peak CRE Portfolio Intelligence: 00 Setup

Entry notebook for the end-to-end build (Lakeflow → Unity Catalog → ML/GenAI → Lakebase → Genie → Databricks App) for the fictional CRE investment manager **Granite Peak Real Estate Partners**. All data is synthetic or public open data.

This notebook creates the schema and the UC volumes that later stages land data into.

**Prerequisites:** Unity Catalog enabled; `USE CATALOG` + `CREATE SCHEMA` on the target catalog; serverless compute.

In [1]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")
dbutils.widgets.dropdown("cleanup", "false", ["false", "true"], "Drop schema at end")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
assert catalog and schema, "Set the 'catalog' and 'schema' widgets before running."
print(f"Target: {catalog}.{schema}")

Target: smmorgan_catalog.granite_commercial_real_estate


In [2]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}` "
          "COMMENT 'Granite Peak CRE Portfolio Intelligence (synthetic + public open data)'")

volumes = {
    "landing": "Raw landing zone: ERP CDC files, lease and deal PDFs, Overture extracts",
    "artifacts": "Generator ground truth and run artifacts",
}
for name, comment in volumes.items():
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{name}` COMMENT '{comment}'")

for sub in ["erp_cdc", "docs/leases", "docs/deals", "overture"]:
    dbutils.fs.mkdirs(f"/Volumes/{catalog}/{schema}/landing/{sub}")

display(spark.sql(f"SHOW VOLUMES IN `{catalog}`.`{schema}`"))

database                       | volume_name
-------------------------------+------------
granite_commercial_real_estate | artifacts  
granite_commercial_real_estate | landing    


In [3]:
if dbutils.widgets.get("cleanup") == "true":
    spark.sql(f"DROP SCHEMA IF EXISTS `{catalog}`.`{schema}` CASCADE")
    print(f"Dropped {catalog}.{schema}")
else:
    print("Cleanup skipped (set the 'cleanup' widget to 'true' to drop the schema).")

Cleanup skipped (set the 'cleanup' widget to 'true' to drop the schema).
